## Phase 1 : RayCluster Setup and Ray Based Distributed Data Processing

- **CodeFlare SDK**: Ray cluster deployment and management on Kubernetes
- **Ray Job Submission**: Distributed synthetic data generation using Ray workers

In [ ]:
# Install CodeFlare SDK
%pip install codeflare-sdk

### Setup Ray Cluster using Codeflare-SDK

Configure and deploy the Ray cluster for distributed data processing


In [ ]:
# Setup Ray cluster using CodeFlare SDK
from codeflare_sdk import Cluster, ClusterConfiguration, TokenAuthentication
import time

token="<auth-token>"
api_server="<api-server-url>"

# Authenticate with the Openshift cluster
auth = TokenAuthentication(
    token=token,
    server=api_server,
    skip_tls=True
)
auth.login()

In [2]:
# Configure Ray cluster for distributed synthetic data generation
from kubernetes.client.models import V1Volume, V1VolumeMount, V1PersistentVolumeClaimVolumeSource

ray_cluster = Cluster(ClusterConfiguration(
    name="test1-cluster",
    num_workers=1,  # 2 workers for distributed processing
    # Head node configuration
    head_cpu_requests=1,
    head_cpu_limits=2,
    head_memory_requests=8,
    head_memory_limits=16,    
    # Worker node configuration  
    worker_cpu_requests=1,
    worker_cpu_limits=2,
    worker_memory_requests=10,
    worker_memory_limits=20,
    # UnComment in case of using accelerators for RayCluster
    # worker_extended_resource_requests={'nvidia.com/gpu': 1},
    # Ray runtime image
    image="quay.io/rhoai/ray:2.35.0-py311-cu121-torch24-fa26",
    # Volume mount - Shared PVC storage with RWX peermissions
    volume_mounts=[
        V1VolumeMount(
            name="shared",
            mount_path="/shared"
        )
    ],
    volumes=[
        V1Volume(
            name="shared",
            persistent_volume_claim=V1PersistentVolumeClaimVolumeSource(
                claim_name="shared"
            )
        )
    ],
))

print(" Ray Cluster Configuration:")
print(f"   Name: {ray_cluster.config.name}")
print(f"   Workers: {ray_cluster.config.num_workers}")
print(f"   Worker Resources: {ray_cluster.config.worker_cpu_requests}CPU, {ray_cluster.config.worker_memory_requests} RAM, {ray_cluster.config.worker_extended_resource_requests} GPU")
print(f"   Image: {ray_cluster.config.image}")


Yaml resources loaded for test1-cluster


 Ray Cluster Configuration:
   Name: test1-cluster
   Workers: 1
   Worker Resources: 1CPU, 10G RAM, {} GPU
   Image: quay.io/rhoai/ray:2.35.0-py311-cu121-torch24-fa26


In [ ]:
# Deploy the Ray cluster
ray_cluster.apply()

In [6]:
# Wait for Ray cluster to be ready
ray_cluster.wait_ready()

Waiting for requested resources to be set up...
Requested cluster is up and running!
Dashboard is ready!


In [7]:
ray_cluster.details()

                        🚀 CodeFlare Cluster Details 🚀                       
                                                                              
 ╭──────────────────────────────────────────────────────────────────────────╮ 
 │   Name                                                                   │ 
 │   test1-cluster                                              Active ✅   │ 
 │                                                                          │ 
 │   URI: ray://<url>.svc:10001              │ 
 │                                                                          │ 
 │   ]8;id=821336;https://\Dashboard🔗]8;;\                                                            │ 
 │                                                                          │ 
 │                       Cluster Resources                                  │ 
 │   ╭── Workers ──╮  ╭───────── Worker specs(each) ─────────╮              │ 
 │   │  # Workers  │  │  Memory      CPU         GPU         │              │ 
 │   │             │  │                                      │              │ 
 │   │  1          │  │  10G~20G     1~2         0           │              │ 
 │   │             │  │                                      │              │ 
 │   ╰─────────────╯  ╰──────────────────────────────────────╯              │ 
 ╰──────────────────────────────────────────────────────────────────────────╯

RayCluster(name='test1-cluster', status=<RayClusterStatus.READY: 'ready'>, head_cpu_requests=1, head_cpu_limits=2, head_mem_requests='8G', head_mem_limits='16G', num_workers=1, worker_mem_requests='10G', worker_mem_limits='20G', worker_cpu_requests=1, worker_cpu_limits=2, namespace='<test-namesace>', dashboard='https://', worker_extended_resources={}, head_extended_resources={})

In [8]:
# Initialize the Job Submission Client
client = ray_cluster.job_client
print("Ray job client initialized")


Ray job client initialized


## Submit Ray Job for Synthetic Data Generation

Submit the synthetic data generation function to the Ray cluster:


In [48]:
# Submit the Ray Data SDG job for distributed synthetic data generation
submission_id = client.submit_job(
    entrypoint="python ray_sdg_job.py --num-cpus 2 --seeds 50 --variations 3 --batch-size 4 --quality-threshold 0.4 --output-path /shared/synthetic_data",
    runtime_env={
        "env_vars": {
            'HF_HOME': '/shared/cache',
            'HF_DATASETS_CACHE': '/shared/cache/datasets',
            'TOKENIZERS_PARALLELISM': 'false',
        },
        'pip': [
            'ray[data]>=2.8.0',
            'transformers>=4.36.0',
            'torch>=2.0.0', 
            'datasets>=2.14.0',
            'accelerate>=0.24.0',
            'numpy>=1.21.0',
            'tqdm>=4.64.0',
            'pyarrow>=12.0.0,<15.0.0',
        ],
        'working_dir': './',
        "excludes": ["*.ipynb", "*.md"]
    },
)

print(f"Ray Data SDG job submitted with ID: {submission_id}")

Ray Data SDG job submitted with ID: raysubmit_qg9j4fzTrYM4Ub5r


In [ ]:
client.get_job_logs(submission_id)

In [47]:
# Stop/Delete any running jobs
# client.stop_job(submission_id)
client.delete_job(submission_id)

True

### Cleanup Ray Cluster

Clean up the Ray cluster resources (following ray_finetune_llm_deepspeed.ipynb pattern):


In [ ]:
# Cleanup Ray cluster (following ray_finetune_llm_deepspeed.ipynb pattern)
print(" Cleaning up Ray cluster...")

# Tear down the Ray cluster
ray_cluster.down()


In [5]:
import os, json
# Check for dataset
paths = ["shared/synthetic_data/synthetic_dataset.json", "shared/synthetic_data/final_synthetic_dataset.json"]
dataset_path = next((p for p in paths if os.path.exists(p)), None)

if dataset_path:
    with open(dataset_path, "r") as f:
        data = json.load(f)
    
    if isinstance(data, list):
        total_samples = len(data)
        avg_quality = sum(item.get('overall_quality', 0) for item in data) / total_samples if total_samples > 0 else 0
        sample = data[0] if data else None
        
        print(f" Dataset found: {total_samples} samples")
        print(f"   Avg quality: {avg_quality:.2f} \n   Source: {sample.get('source', 'N/A') if sample else 'N/A'}")
    
    # Show sample
    if sample:
        print(f"   Sample Question -> {sample['question']}")
        print(f"   Sample Answer -> {sample['answer']}")
    
    print("\n Ready for training!")    
else:
    print(" Dataset not found. Run Ray Data job first.")

 Dataset found: 31 samples
   Avg quality: 0.38 
   Source: ray_data_sdg_qwen
   Sample Question -> A school bought 50 pencils for its students. After a week, it gave away 10 pencils. Later, it received an additional shipment of 30 pencils. How many pencils does the school have now?
   Sample Answer -> The school initially had 50 pencils. It gave away 10 pencils, so it had 50 - 10 = 40 pencils left. Then, it received an additional 30 pencils, so it has 40 + 30 = 70 pencils now.

 Ready for training!
